In [ ]:
import cv2
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torchvision import transforms
import os
import random
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import pandas as pd




In [ ]:
import kagglehub

# Download latest version
base_path = kagglehub.dataset_download("davilsena/ckdataset")

print("Path to dataset files:", base_path)

100%|██████████| 2.48M/2.48M [00:00<00:00, 55.3MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/davilsena/ckdataset/versions/2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

save_dir = "/content/drive/MyDrive/fer_emotion"
os.makedirs(save_dir, exist_ok=True)  # create if not exists

save_path = os.path.join(save_dir, "checkpoint.pth")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

print("Contents of base_path:")
print(os.listdir(base_path))


Contents of base_path:
['ckextended.csv']


In [ ]:
import pandas as pd

csv_file = f"{base_path}/ckextended.csv"  # adjust path if needed
data = pd.read_csv(csv_file)

print("Columns in CSV:")
print(data.columns.tolist())


Columns in CSV:
['emotion', 'pixels', 'Usage']


In [ ]:
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
import numpy as np
import cv2
from torchvision import transforms


class CKPlusDualFilterDataset(Dataset):
    def __init__(self, csv_file=None, dataframe=None, transform=None, usage=None, image_size=(224, 224)):
        """
        CKPlusDualFilterDataset loads CK+ data from a CSV or DataFrame.
        Each sample produces a 4-channel tensor: [R, G, B, Edge].
        """
        if dataframe is not None:
            self.data = dataframe.copy()
        elif csv_file is not None:
            self.data = pd.read_csv(csv_file)
        else:
            raise ValueError("Either csv_file or dataframe must be provided")

        # Optional filtering by Usage column
        if usage and 'Usage' in self.data.columns:
            self.data = self.data[self.data['Usage'] == usage]
            print(f"Filtered for '{usage}': {len(self.data)} samples")

        self.transform = transform
        self.image_size = image_size

        # Encode emotion labels
        self.classes = sorted(self.data['emotion'].unique())
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}

        # Define resize/ToTensor transformations
        self.basic_transform = transforms.Compose([
            transforms.Resize(self.image_size),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # --- Load pixels and label ---
        pixel_str = self.data.iloc[idx]['pixels']
        img_array = np.array([int(p) for p in pixel_str.split()], dtype=np.uint8).reshape(48, 48)
        label = self.class_to_idx[self.data.iloc[idx]['emotion']]

        # --- Convert grayscale → RGB ---
        rgb_img = cv2.cvtColor(img_array, cv2.COLOR_GRAY2RGB)

        # --- Generate Canny edge map ---
        blurred = cv2.GaussianBlur(img_array, (3, 3), 0)
        edges = cv2.Canny(blurred, 80, 160)

        # --- Convert to PIL Images ---
        rgb_img = Image.fromarray(rgb_img)
        edge_img = Image.fromarray(edges)

        # --- Apply resize + tensor conversion individually ---
        rgb_tensor = self.basic_transform(rgb_img)
        edge_tensor = self.basic_transform(edge_img)

        # --- Stack [3 RGB + 1 Edge] → [4, H, W] ---
        stacked = torch.cat([rgb_tensor, edge_tensor], dim=0)

        # --- Normalize manually to [-1, 1] range ---
        stacked = (stacked - 0.5) / 0.5

        return stacked, label


In [ ]:
import os
import torch
import pandas as pd
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

# --- Hyperparameters ---
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 100
HIDDEN_SIZE = 256
LSTM_LAYERS = 4
LEARNING_RATE = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- FIXED: Transformations for 4-channel images ---
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),                     # Resize to target size
    transforms.RandomHorizontalFlip(),                # Flip images randomly
    transforms.RandomRotation(15),                    # Rotate randomly ±15 degrees
    transforms.ColorJitter(brightness=0.2,
                           contrast=0.2,
                           saturation=0.2,
                           hue=0.1),                  # Adjust color slightly (RGB channels)
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5, 0.5],  # Normalize 4 channels
                         std=[0.5, 0.5, 0.5, 0.5]),
])

test_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5, 0.5]),
])

# --- Load CK+ CSV ---
csv_path = os.path.join(base_path, "ckextended.csv")
ck_df = pd.read_csv(csv_path)
print(f"Dataset size: {len(ck_df)}")
print(f"Usage distribution:\n{ck_df['Usage'].value_counts()}")

# --- Stratified train-test split ---
train_df, test_df = train_test_split(
    ck_df, test_size=0.2, random_state=42, stratify=ck_df['emotion']
)

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

# --- Load Dual-Filter Dataset ---
train_dataset = CKPlusDualFilterDataset(dataframe=train_df, transform=train_transform)
test_dataset = CKPlusDualFilterDataset(dataframe=test_df, transform=test_transform)

# --- Update NUM_CLASSES dynamically ---
NUM_CLASSES = len(train_dataset.classes)
print(f"Training with {NUM_CLASSES} classes: {train_dataset.classes}")

# --- DataLoaders ---
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# --- Test sample from dataset ---
print("\nTesting dataset sample...")
img, label = train_dataset[0]
print(f"Sample image shape (dual-filter stack): {img.shape}")  # Should be [4, H, W]
print(f"Sample label: {label}")

print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")

Dataset size: 920
Usage distribution:
Usage
Training       734
PrivateTest     95
PublicTest      91
Name: count, dtype: int64
Training samples: 736
Test samples: 184
Training with 8 classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]

Testing dataset sample...
Sample image shape (dual-filter stack): torch.Size([4, 224, 224])
Sample label: 6
Train samples: 736, Test samples: 184
Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]


In [ ]:
print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")


Train samples: 736, Test samples: 184
Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

# ---------- Residual + SE block ----------
class ResidualSEBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.4, reduction=16):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.dropout = nn.Dropout2d(dropout)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(out_channels, max(out_channels // reduction, 4), 1),
            nn.GELU(),
            nn.Conv2d(max(out_channels // reduction, 4), out_channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.dropout(self.bn2(self.conv2(out)))
        out += self.shortcut(x)
        out = F.relu(out)
        se_weight = self.se(out)
        return out * se_weight

# ---------- Local Feature Extractor ----------
class LocalFeatureExtractor(nn.Module):
    def __init__(self, in_channels=3, base_channels=32, dropout=0.4, out_dim=128):
        super().__init__()
        self.layer1 = ResidualSEBlock(in_channels, base_channels)
        self.layer2 = ResidualSEBlock(base_channels, base_channels*2)
        self.layer3 = ResidualSEBlock(base_channels*2, base_channels*4)
        self.layer4 = ResidualSEBlock(base_channels*4, base_channels*8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(base_channels*8, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)  # [B, out_dim]

# ---------- Global Feature Extractor with ResNet101 + EfficientNet ----------
class GlobalFeatureExtractor(nn.Module):
    def __init__(self, in_channels=3, output_dim=256, dropout=0.4, use_pretrained=True):
        super().__init__()

        # ---- ResNet101 ----
        self.resnet = models.resnet101(weights=(models.ResNet101_Weights.IMAGENET1K_V2 if use_pretrained else None))
        orig_conv = self.resnet.conv1
        if in_channels != 3:
            new_conv = nn.Conv2d(in_channels, orig_conv.out_channels, kernel_size=orig_conv.kernel_size,
                                 stride=orig_conv.stride, padding=orig_conv.padding, bias=False)
            if use_pretrained:
                with torch.no_grad():
                    if in_channels == 1:
                        w = orig_conv.weight.data.mean(dim=1, keepdim=True)
                        new_conv.weight.data = w
                    else:
                        w = orig_conv.weight.data.mean(dim=1, keepdim=True).repeat(1, in_channels, 1, 1)
                        new_conv.weight.data = w
            self.resnet.conv1 = new_conv
        self.resnet_features = nn.Sequential(*list(self.resnet.children())[:-2])
        self.resnet_pool = nn.AdaptiveAvgPool2d(1)

        # ---- EfficientNet B0 ----
        self.efficientnet = models.efficientnet_b0(weights=(models.EfficientNet_B0_Weights.IMAGENET1K_V1 if use_pretrained else None))
        if in_channels != 3:
            orig_conv = self.efficientnet.features[0][0]
            new_conv = nn.Conv2d(in_channels, orig_conv.out_channels, kernel_size=orig_conv.kernel_size,
                                 stride=orig_conv.stride, padding=orig_conv.padding, bias=False)
            if use_pretrained:
                with torch.no_grad():
                    if in_channels == 1:
                        new_conv.weight.data = orig_conv.weight.data.mean(dim=1, keepdim=True)
                    else:
                        new_conv.weight.data = orig_conv.weight.data.mean(dim=1, keepdim=True).repeat(1, in_channels, 1, 1)
            self.efficientnet.features[0][0] = new_conv
        self.efficientnet_pool = nn.AdaptiveAvgPool2d(1)

        # ---- Fully connected for fusion ----
        self.fc = nn.Sequential(
            nn.Linear(2048 + 1280, 512),  # ResNet101=2048, EfficientNetB0=1280
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, output_dim),
            nn.BatchNorm1d(output_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # ResNet101 features
        f_res = self.resnet_features(x)
        f_res = self.resnet_pool(f_res).flatten(1)

        # EfficientNet features
        f_eff = self.efficientnet.features(x)
        f_eff = self.efficientnet_pool(f_eff).flatten(1)

        # Concatenate and project
        f = torch.cat([f_res, f_eff], dim=1)
        return self.fc(f)  # [B, output_dim]

# ---------- Dual Transformer Fusion Model ----------
class DualTransformerFusionModel(nn.Module):
    def __init__(self, num_emotions=8, proj_dim=256, n_heads=4, n_layers=1, dropout=0.4, use_pretrained=True):
        super().__init__()

        # Separate extractors for each modality
        self.local_extractor_rgb = LocalFeatureExtractor(in_channels=3, base_channels=32, dropout=dropout, out_dim=128)
        self.global_extractor_rgb = GlobalFeatureExtractor(in_channels=3, output_dim=256, dropout=dropout, use_pretrained=use_pretrained)

        self.local_extractor_canny = LocalFeatureExtractor(in_channels=3, base_channels=32, dropout=dropout, out_dim=128)
        self.global_extractor_canny = GlobalFeatureExtractor(in_channels=3, output_dim=256, dropout=dropout, use_pretrained=use_pretrained)

        # Projection layers
        self.local_proj = nn.Linear(128, proj_dim)
        self.global_proj = nn.Linear(256, proj_dim)

        # Transformer fusion
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=proj_dim, nhead=n_heads, dim_feedforward=proj_dim*2, dropout=dropout, batch_first=True
        )
        self.modality_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim * 2, 512),  # Increased capacity
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_emotions)
        )

    def forward(self, rgb, canny):
        # RGB features
        local_rgb = self.local_proj(self.local_extractor_rgb(rgb))
        global_rgb = self.global_proj(self.global_extractor_rgb(rgb))
        rgb_features = local_rgb + global_rgb

        # Canny features
        local_canny = self.local_proj(self.local_extractor_canny(canny))
        global_canny = self.global_proj(self.global_extractor_canny(canny))
        canny_features = local_canny + global_canny

        # Concatenate features
        fused_features = torch.cat([rgb_features, canny_features], dim=1)

        # Classification
        logits = self.classifier(fused_features)
        return logits


In [ ]:
model = DualTransformerFusionModel(num_emotions=NUM_CLASSES).to(device)


Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:00<00:00, 191MB/s]


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 158MB/s]


In [ ]:
import pandas as pd
import torch

# Choose your model
model_to_inspect = model_shared  # or model

# Define major blocks manually (adjust based on your model)
blocks = {
    "Local Extractor": model_to_inspect.local_extractor,
    "Global Extractor": model_to_inspect.global_extractor,
    "Local Projection": model_to_inspect.local_proj,
    "Global Projection": model_to_inspect.global_proj,
    "Transformer Encoder": model_to_inspect.modality_encoder,
    "Classifier": model_to_inspect.classifier
}

# Collect total parameters per block
summary_data = []
for block_name, block in blocks.items():
    total_params = sum(p.numel() for p in block.parameters())
    summary_data.append({
        "Layer / Block": block_name,
        "Total Parameters": total_params
    })

# Create DataFrame
summary_df = pd.DataFrame(summary_data)

# Display table
print(summary_df.to_string(index=False))

# Optional: save CSV for Word
summary_df.to_csv("model_summary_parameters.csv", index=False)


      Layer / Block  Total Parameters
    Local Extractor           1261952
   Global Extractor          51675020
   Local Projection             33024
  Global Projection             65792
Transformer Encoder            527104
         Classifier            397576


In [ ]:



import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchview import draw_graph

# ---------- Residual + SE block ----------
class ResidualSEBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.4, reduction=16):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.dropout = nn.Dropout2d(dropout)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(out_channels, max(out_channels // reduction, 4), 1),
            nn.GELU(),
            nn.Conv2d(max(out_channels // reduction, 4), out_channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.dropout(self.bn2(self.conv2(out)))
        out += self.shortcut(x)
        out = F.relu(out)
        se_weight = self.se(out)
        return out * se_weight


# ---------- Local Feature Extractor ----------
class LocalFeatureExtractor(nn.Module):
    def __init__(self, in_channels=3, base_channels=32, dropout=0.4, out_dim=128):
        super().__init__()
        self.layer1 = ResidualSEBlock(in_channels, base_channels)
        self.layer2 = ResidualSEBlock(base_channels, base_channels*2)
        self.layer3 = ResidualSEBlock(base_channels*2, base_channels*4)
        self.layer4 = ResidualSEBlock(base_channels*4, base_channels*8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(base_channels*8, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


# ---------- Global Feature Extractor ----------
class GlobalFeatureExtractor(nn.Module):
    def __init__(self, in_channels=3, output_dim=256, dropout=0.4, use_pretrained=False):
        super().__init__()

        self.resnet = models.resnet101(weights=None)
        self.resnet_features = nn.Sequential(*list(self.resnet.children())[:-2])
        self.resnet_pool = nn.AdaptiveAvgPool2d(1)

        self.efficientnet = models.efficientnet_b0(weights=None)
        self.efficientnet_pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(2048 + 1280, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, output_dim),
            nn.BatchNorm1d(output_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        f_res = self.resnet_pool(self.resnet_features(x)).flatten(1)
        f_eff = self.efficientnet_pool(self.efficientnet.features(x)).flatten(1)
        f = torch.cat([f_res, f_eff], dim=1)
        return self.fc(f)


# ---------- Dual Transformer Fusion Model ----------
class DualTransformerFusionModelShared(nn.Module):
    def __init__(self, num_emotions=8, proj_dim=256, n_heads=4, n_layers=1, dropout=0.4):
        super().__init__()

        # Shared feature extractors
        self.local_extractor = LocalFeatureExtractor()
        self.global_extractor = GlobalFeatureExtractor()

        self.local_proj = nn.Linear(128, proj_dim)
        self.global_proj = nn.Linear(256, proj_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=proj_dim,
            nhead=n_heads,
            dim_feedforward=proj_dim * 2,
            dropout=dropout,
            batch_first=True
        )
        self.modality_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.classifier = nn.Sequential(
            nn.Linear(proj_dim * 2, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_emotions)
        )

    def forward(self, rgb, canny):
        # Shared extractor used for both inputs
        local_rgb = self.local_proj(self.local_extractor(rgb))
        global_rgb = self.global_proj(self.global_extractor(rgb))
        rgb_features = local_rgb + global_rgb

        local_canny = self.local_proj(self.local_extractor(canny))
        global_canny = self.global_proj(self.global_extractor(canny))
        canny_features = local_canny + global_canny

        fused = torch.cat([rgb_features, canny_features], dim=1)
        return self.classifier(fused)


# Instantiate
model_shared = DualTransformerFusionModelShared()
model_shared.eval()

dummy_rgb = torch.randn(1, 3, 224, 224)
dummy_canny = torch.randn(1, 3, 224, 224)

# Draw graph (single shared branch)
graph = draw_graph(
    model_shared,
    input_data=(dummy_rgb, dummy_canny),
    expand_nested=True,
    show_shapes=True,
    graph_name="Dual Transformer Fusion Model (Shared Extractors)"
)

# Graph settings remain the same
graph.visual_graph.graph_attr.update({
    "rankdir": "TB",
    "dpi": "300",
    "size": "8,16!",
    "ratio": "compress",
    "nodesep": "0.25",
    "ranksep": "0.35",
    "fontname": "Helvetica"
})

graph.visual_graph.node_attr.update({
    "shape": "box",
    "style": "rounded,filled",
    "fillcolor": "lightgray",
    "fontsize": "8",
    "margin": "0.04,0.02",
    "fontname": "Helvetica"
})

graph.visual_graph.edge_attr.update({
    "arrowsize": "0.5"
})

graph.visual_graph.render(
    filename="dual_transformer_shared_branch",
    directory="/content",
    format="png",
    cleanup=True
)







'/content/dual_transformer_shared_branch.png'

In [ ]:
import torch
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
import os
import cv2
import numpy as np

# --------------------------- #
# 🚀 Training Function (with AMP + Gradient Clipping) - FIXED FOR DUAL INPUT
# --------------------------- #
def train_one_epoch(model, dataloader, criterion, optimizer, device, scaler, max_grad_norm=1.0):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader, desc="Training", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        # Extract RGB channels
        rgb = images[:, :3, :, :]

        # Generate proper Canny edges from RGB (convert to grayscale first)
        batch_size = images.size(0)
        canny_maps = []

        with torch.no_grad():
            for i in range(batch_size):
                # Convert RGB to grayscale using standard weights
                # RGB tensor is in range [-1, 1] due to normalization, so convert to [0, 255]
                rgb_denorm = ((rgb[i] * 0.5) + 0.5) * 255.0  # [-1,1] -> [0,255]
                rgb_denorm = rgb_denorm.byte().cpu().numpy().transpose(1, 2, 0)

                # Convert to grayscale
                gray = cv2.cvtColor(rgb_denorm, cv2.COLOR_RGB2GRAY)

                # Generate Canny edges
                blurred = cv2.GaussianBlur(gray, (3, 3), 0)
                edges = cv2.Canny(blurred, 80, 160)

                # Convert back to tensor and normalize to [-1, 1]
                edges_tensor = torch.from_numpy(edges).float().unsqueeze(0)  # [1, H, W]
                edges_tensor = (edges_tensor / 255.0 - 0.5) / 0.5  # [0,255] -> [-1,1]
                canny_maps.append(edges_tensor)

            # Stack all canny maps and move to device
            canny = torch.stack(canny_maps).to(device)
            # Convert to 3-channel by repeating the single channel
            canny_3ch = canny.repeat(1, 3, 1, 1)

        # Automatic Mixed Precision (AMP)
        with autocast():
            outputs = model(rgb, canny_3ch)  # forward pass with proper inputs
            loss = criterion(outputs, labels)

        # Backpropagation with gradient scaling
        scaler.scale(loss).backward()

        # Gradient clipping for stability
        if max_grad_norm is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        scaler.step(optimizer)
        scaler.update()

        # Metrics
        running_loss += loss.item() * labels.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    accuracy = correct / total
    return epoch_loss, accuracy


# --------------------------- #
# 🧠 Validation Function (with AMP) - FIXED FOR DUAL INPUT
# --------------------------- #
def validate(model, dataloader, criterion, device):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Validating", leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # Extract RGB channels
            rgb = images[:, :3, :, :]

            # Generate proper Canny edges from RGB (same as training)
            batch_size = images.size(0)
            canny_maps = []

            for i in range(batch_size):
                # Convert RGB to grayscale
                rgb_denorm = ((rgb[i] * 0.5) + 0.5) * 255.0  # [-1,1] -> [0,255]
                rgb_denorm = rgb_denorm.byte().cpu().numpy().transpose(1, 2, 0)

                gray = cv2.cvtColor(rgb_denorm, cv2.COLOR_RGB2GRAY)

                # Generate Canny edges
                blurred = cv2.GaussianBlur(gray, (3, 3), 0)
                edges = cv2.Canny(blurred, 80, 160)

                # Convert back to tensor and normalize
                edges_tensor = torch.from_numpy(edges).float().unsqueeze(0)
                edges_tensor = (edges_tensor / 255.0 - 0.5) / 0.5
                canny_maps.append(edges_tensor)

            canny = torch.stack(canny_maps).to(device)
            canny_3ch = canny.repeat(1, 3, 1, 1)

            with autocast():
                outputs = model(rgb, canny_3ch)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * labels.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    epoch_loss = val_loss / total
    accuracy = correct / total
    return epoch_loss, accuracy



In [ ]:
start_epoch = 0
best_val_acc = 0.0

if os.path.exists(save_path):
    print("🔄 Loading checkpoint...")
    checkpoint = torch.load(save_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch']
    best_val_acc = checkpoint['val_acc']
    print(f"Resumed from epoch {start_epoch}, best val acc: {best_val_acc:.4f}")


In [ ]:
import os
import json
import pickle
import torch

# --------------------------- #
# 📁 Training Metrics Setup
# --------------------------- #
metrics_path = os.path.join(save_dir, "training_metrics.pkl")
metrics_json_path = os.path.join(save_dir, "training_metrics.json")

# Load or initialize metrics
if os.path.exists(metrics_path):
    print("📊 Loading previous training metrics...")
    with open(metrics_path, 'rb') as f:
        metrics = pickle.load(f)
    train_losses = metrics.get('train_losses', [])
    val_losses = metrics.get('val_losses', [])
    train_accs = metrics.get('train_accs', [])
    val_accs = metrics.get('val_accs', [])
    learning_rates = metrics.get('learning_rates', [])
else:
    print("🚀 Starting new training metrics...")
    train_losses, val_losses, train_accs, val_accs, learning_rates = [], [], [], [], []


# --------------------------- #
# 💾 Save Metrics Function
# --------------------------- #
def save_training_metrics(train_losses, val_losses, train_accs, val_accs, learning_rates):
    """Save training metrics to disk (pickle + JSON)."""
    metrics = {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accs': train_accs,
        'val_accs': val_accs,
        'learning_rates': learning_rates
    }

    # Save Pickle (for resume)
    with open(metrics_path, 'wb') as f:
        pickle.dump(metrics, f)

    # Save JSON (for human readability)
    with open(metrics_json_path, 'w') as f:
        json.dump(metrics, f, indent=2)

    print("💾 Training metrics saved!")


# --------------------------- #
# 🔍 Pre-Training Verification
# --------------------------- #
print("🔍 Final verification before training...")
print(f"Model device: {next(model.parameters()).device}")
print(f"Train loader batches: {len(train_loader)}")
print(f"Test loader batches: {len(test_loader)}")
print(f"Number of classes: {NUM_CLASSES}")

# ✅ FIXED Forward Pass Test (Dual-input model)
# ✅ FIXED Forward Pass Test (Dual-input model)
model.eval()
with torch.no_grad():
    images, labels = next(iter(train_loader))
    images = images.to(device, non_blocking=True)

    # Split into RGB and Canny parts
    rgb = images[:, :3, :, :]
    canny = images[:, 3:, :, :].repeat(1, 3, 1, 1)  # <-- FIX: repeat single-channel to 3 channels

    outputs = model(rgb, canny)

    print(f"Input batch shape: {images.shape}")
    print(f"RGB shape: {rgb.shape}, Canny shape: {canny.shape}")
    print(f"Model output shape: {outputs.shape}")
    print(f"Batch label range: {labels.min()} → {labels.max()}")

print("✅ Model forward pass test successful. Starting training...\n")



# --------------------------- #
# 🧠 Training Loop
# --------------------------- #
for epoch in range(start_epoch, EPOCHS):
    print(f"\n🌟 Epoch {epoch+1}/{EPOCHS}")

    # Training phase
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler, max_grad_norm=1.0)

    # Validation phase
    val_loss, val_acc = validate(model, test_loader, criterion, device)

    # Scheduler step
    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]['lr']

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    learning_rates.append(current_lr)

    # Log progress
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print(f"Learning Rate: {current_lr:.2e}")

    # Save metrics each epoch
    save_training_metrics(train_losses, val_losses, train_accs, val_accs, learning_rates)

    # Save best checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'val_acc': best_val_acc,
            'train_acc': train_acc,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'train_accs': train_accs,
            'val_accs': val_accs
        }, save_path)
        print("✅ Saved new best checkpoint!")

    # Optional early stopping
    if current_lr < 1e-7:
        print("🛑 Learning rate too low, stopping training...")
        break

print("\n🎉 Training Complete!")
print(f"🏆 Best validation accuracy: {best_val_acc:.4f}")

# Optional: generate training plots later
print("\n📊 Generating training plots...")


🚀 Starting new training metrics...
🔍 Final verification before training...
Model device: cuda:0
Train loader batches: 147
Test loader batches: 37
Number of classes: 7
Input batch shape: torch.Size([16, 4, 224, 224])
Temporal stack shape: torch.Size([16, 2, 3, 224, 224])
Emotion output shape: torch.Size([16, 7])
Valence/Arousal output shape: torch.Size([16, 2])
AUs output shape: torch.Size([16, 17])
Batch label range: 0 → 6

🌟 Epoch 1/100


Training:   0%|          | 0/147 [00:00<?, ?it/s]/tmp/ipython-input-934693970.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Validating:   0%|          | 0/37 [00:00<?, ?it/s]/tmp/ipython-input-934693970.py:63: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Train Loss: 1.9221 | Train Acc: 0.2064
Val   Loss: nan | Val   Acc: 0.5102
Learning Rate: 1.00e-04
💾 Training metrics saved!
✅ Saved new best checkpoint!

🌟 Epoch 2/100


Train Loss: 1.3416 | Train Acc: 0.5302
Val   Loss: 0.9319 | Val   Acc: 0.7296
Learning Rate: 1.00e-04
💾 Training metrics saved!
✅ Saved new best checkpoint!

🌟 Epoch 3/100


Train Loss: 0.9112 | Train Acc: 0.7566
Val   Loss: nan | Val   Acc: 0.8163
Learning Rate: 1.00e-04
💾 Training metrics saved!
✅ Saved new best checkpoint!

🌟 Epoch 4/100


...

Train Loss: 0.0382 | Train Acc: 0.9974
Val   Loss: 0.5121 | Val   Acc: 0.9218
Learning Rate: 5.00e-05
💾 Training metrics saved!

🌟 Epoch 24/100


Train Loss: 0.0417 | Train Acc: 0.9957
Val   Loss: 0.5574 | Val   Acc: 0.9150
Learning Rate: 5.00e-05
💾 Training metrics saved!

🌟 Epoch 25/100


Train Loss: 0.0343 | Train Acc: 0.9970
Val   Loss: 0.4982 | Val   Acc: 0.9235
Learning Rate: 5.00e-05
💾 Training metrics saved!

🌟 Epoch 26/100


Train Loss: 0.0413 | Train Acc: 0.9945
Val   Loss: 0.4817 | Val   Acc: 0.9303
Learning Rate: 5.00e-05
💾 Training metrics saved!
✅ Saved new best checkpoint!

🌟 Epoch 27/100


Training:  93%|█████████▎| 136/147 [02:49<00:13,  1.25s/it]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, classification_report, precision_recall_fscore_support,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Set style for better plots
plt.style.use('default')
sns.set_palette("husl")

# Define emotion classes based on CK+ dataset
# CK+ emotions: 0=anger, 1=contempt, 2=disgust, 3=fear, 4=happy, 5=sadness, 6=surprise
class_names = ['anger', 'contempt', 'disgust', 'fear', 'happy', 'sadness', 'surprise', 'neutral']
# Adjust based on your actual classes
class_names = class_names[:NUM_CLASSES]

def plot_training_history(train_losses, val_losses, train_accs, val_accs, learning_rates):
    """Plot comprehensive training history"""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

    epochs = range(1, len(train_losses) + 1)

    # Plot 1: Loss
    ax1.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Plot 2: Accuracy
    ax2.plot(epochs, train_accs, 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, val_accs, 'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # Plot 3: Learning Rate
    if learning_rates:
        ax3.semilogy(epochs, learning_rates, 'g-', linewidth=2)
        ax3.set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('Learning Rate (log scale)')
        ax3.grid(True, alpha=0.3)

    # Plot 4: Combined view
    best_epoch = np.argmax(val_accs) + 1
    best_val_acc = np.max(val_accs)

    ax4.plot(epochs, train_accs, 'b-', label='Train Acc', linewidth=2)
    ax4.plot(epochs, val_accs, 'r-', label='Val Acc', linewidth=2)
    ax4.axvline(x=best_epoch, color='green', linestyle='--',
                label=f'Best Epoch: {best_epoch}\nBest Acc: {best_val_acc:.4f}')

    ax4_twin = ax4.twinx()
    ax4_twin.plot(epochs, train_losses, 'b--', alpha=0.7, label='Train Loss')
    ax4_twin.plot(epochs, val_losses, 'r--', alpha=0.7, label='Val Loss')

    ax4.set_title('Training Overview', fontsize=14, fontweight='bold')
    ax4.set_xlabel('Epoch')
    ax4.set_ylabel('Accuracy', color='black')
    ax4_twin.set_ylabel('Loss', color='black')

    lines1, labels1 = ax4.get_legend_handles_labels()
    lines2, labels2 = ax4_twin.get_legend_handles_labels()
    ax4.legend(lines1 + lines2, labels1 + labels2, loc='center right')
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'training_history.png'), dpi=300, bbox_inches='tight')
    plt.show()

    return best_epoch, best_val_acc

def get_all_predictions(model, dataloader, device):
    """Get all predictions and true labels"""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Collecting predictions"):
            images = images.to(device)
            outputs = model(images)
            probs = F.softmax(outputs["emotion"], dim=1)
            _, preds = torch.max(outputs["emotion"], 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

def plot_confusion_matrix(true_labels, predictions, class_names):
    """Plot detailed confusion matrix"""
    cm = confusion_matrix(true_labels, predictions)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'shrink': 0.8})

    plt.title('Confusion Matrix', fontsize=16, fontweight='bold', pad=20)
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
    plt.show()

    return cm

def plot_normalized_confusion_matrix(true_labels, predictions, class_names):
    """Plot normalized confusion matrix"""
    cm = confusion_matrix(true_labels, predictions)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'shrink': 0.8})

    plt.title('Normalized Confusion Matrix', fontsize=16, fontweight='bold', pad=20)
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'confusion_matrix_normalized.png'), dpi=300, bbox_inches='tight')
    plt.show()

    return cm_normalized

def calculate_detailed_metrics(true_labels, predictions, class_names):
    """Calculate and display comprehensive metrics"""
    # Classification report
    report = classification_report(true_labels, predictions,
                                  target_names=class_names, output_dict=True)

    # Precision, recall, f1 for each class
    precision, recall, f1, support = precision_recall_fscore_support(
        true_labels, predictions, average=None
    )

    # Overall metrics
    accuracy = np.mean(true_labels == predictions)
    macro_f1 = f1.mean()
    weighted_f1 = np.average(f1, weights=support)

    # Create detailed metrics dataframe
    metrics_df = pd.DataFrame({
        'Class': class_names,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Support': support
    })

    # Add overall metrics
    overall_metrics = pd.DataFrame({
        'Class': ['Overall'],
        'Precision': [precision.mean()],
        'Recall': [recall.mean()],
        'F1-Score': [macro_f1],
        'Support': [len(true_labels)]
    })

    metrics_df = pd.concat([metrics_df, overall_metrics], ignore_index=True)

    print("=" * 60)
    print("DETAILED CLASSIFICATION METRICS")
    print("=" * 60)
    print(f"Overall Accuracy: {accuracy:.4f}")
    print(f"Macro F1-Score: {macro_f1:.4f}")
    print(f"Weighted F1-Score: {weighted_f1:.4f}")
    print("\nPer-class metrics:")
    print(metrics_df.to_string(index=False))

    # Save to file
    metrics_df.to_csv(os.path.join(save_dir, 'detailed_metrics.csv'), index=False)

    return metrics_df, report

def plot_roc_curves(true_labels, probabilities, class_names):
    """Plot ROC curves for all classes"""
    # Binarize labels for ROC curve
    y_true_bin = label_binarize(true_labels, classes=range(len(class_names)))

    # Compute ROC curve and ROC area for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for i in range(len(class_names)):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], probabilities[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Compute micro-average ROC curve and ROC area
    fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), probabilities.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    # Plot all ROC curves
    plt.figure(figsize=(10, 8))

    colors = plt.cm.rainbow(np.linspace(0, 1, len(class_names)))
    for i, color in zip(range(len(class_names)), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2,
                 label=f'{class_names[i]} (AUC = {roc_auc[i]:.3f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curves - Multi-class', fontsize=16, fontweight='bold')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'roc_curves.png'), dpi=300, bbox_inches='tight')
    plt.show()

    return roc_auc

def plot_precision_recall_curves(true_labels, probabilities, class_names):
    """Plot Precision-Recall curves for all classes"""
    # Binarize labels
    y_true_bin = label_binarize(true_labels, classes=range(len(class_names)))

    # Compute Precision-Recall curve and area for each class
    precision = dict()
    recall = dict()
    average_precision = dict()

    for i in range(len(class_names)):
        precision[i], recall[i], _ = precision_recall_curve(y_true_bin[:, i], probabilities[:, i])
        average_precision[i] = average_precision_score(y_true_bin[:, i], probabilities[:, i])

    # Plot all Precision-Recall curves
    plt.figure(figsize=(10, 8))

    colors = plt.cm.rainbow(np.linspace(0, 1, len(class_names)))
    for i, color in zip(range(len(class_names)), colors):
        plt.plot(recall[i], precision[i], color=color, lw=2,
                 label=f'{class_names[i]} (AP = {average_precision[i]:.3f})')

    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.title('Precision-Recall Curves - Multi-class', fontsize=16, fontweight='bold')
    plt.legend(loc="lower left")
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'precision_recall_curves.png'), dpi=300, bbox_inches='tight')
    plt.show()

    return average_precision

def plot_class_distribution(true_labels, predictions, class_names):
    """Plot class distribution and prediction distribution"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # True distribution
    true_counts = np.bincount(true_labels, minlength=len(class_names))
    ax1.bar(range(len(class_names)), true_counts, color='skyblue', alpha=0.7)
    ax1.set_title('True Class Distribution', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Class')
    ax1.set_ylabel('Count')
    ax1.set_xticks(range(len(class_names)))
    ax1.set_xticklabels(class_names, rotation=45)

    # Add count labels on bars
    for i, count in enumerate(true_counts):
        ax1.text(i, count + max(true_counts)*0.01, str(count),
                ha='center', va='bottom', fontweight='bold')

    # Predicted distribution
    pred_counts = np.bincount(predictions, minlength=len(class_names))
    ax2.bar(range(len(class_names)), pred_counts, color='lightcoral', alpha=0.7)
    ax2.set_title('Predicted Class Distribution', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Class')
    ax2.set_ylabel('Count')
    ax2.set_xticks(range(len(class_names)))
    ax2.set_xticklabels(class_names, rotation=45)

    # Add count labels on bars
    for i, count in enumerate(pred_counts):
        ax2.text(i, count + max(pred_counts)*0.01, str(count),
                ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'class_distribution.png'), dpi=300, bbox_inches='tight')
    plt.show()

def create_comprehensive_report(model, test_loader, device, class_names):
    """Create comprehensive evaluation report"""
    print("🚀 Starting comprehensive evaluation...")

    # Get all predictions
    true_labels, predictions, probabilities = get_all_predictions(model, test_loader, device)

    # 1. Training history
    print("\n📈 1. Training History")
    best_epoch, best_val_acc = plot_training_history(
        train_losses, val_losses, train_accs, val_accs, learning_rates
    )

    # 2. Confusion matrices
    print("\n🎯 2. Confusion Matrices")
    cm = plot_confusion_matrix(true_labels, predictions, class_names)
    cm_normalized = plot_normalized_confusion_matrix(true_labels, predictions, class_names)

    # 3. Detailed metrics
    print("\n📊 3. Detailed Metrics")
    metrics_df, classification_report_dict = calculate_detailed_metrics(
        true_labels, predictions, class_names
    )

    # 4. ROC curves
    print("\n📈 4. ROC Analysis")
    roc_auc = plot_roc_curves(true_labels, probabilities, class_names)

    # 5. Precision-Recall curves
    print("\n🎯 5. Precision-Recall Analysis")
    average_precision = plot_precision_recall_curves(true_labels, probabilities, class_names)

    # 6. Class distribution
    print("\n📋 6. Class Distribution Analysis")
    plot_class_distribution(true_labels, predictions, class_names)

    # Create final summary
    summary = {
        'best_epoch': best_epoch,
        'best_validation_accuracy': best_val_acc,
        'test_accuracy': np.mean(true_labels == predictions),
        'macro_f1': metrics_df.iloc[-1]['F1-Score'],
        'confusion_matrix': cm.tolist(),
        'roc_auc_scores': roc_auc,
        'average_precision_scores': average_precision,
        'class_names': class_names,
        'total_test_samples': len(true_labels)
    }

    # Save summary
    with open(os.path.join(save_dir, 'evaluation_summary.json'), 'w') as f:
        json.dump(summary, f, indent=2)

    print("\n" + "="*60)
    print("🎉 COMPREHENSIVE EVALUATION COMPLETE!")
    print("="*60)
    print(f"📁 All results saved to: {save_dir}")
    print(f"🏆 Best Validation Accuracy: {best_val_acc:.4f}")
    print(f"🧪 Test Accuracy: {summary['test_accuracy']:.4f}")
    print(f"📊 Macro F1-Score: {summary['macro_f1']:.4f}")
    print(f"📈 Mean ROC AUC: {np.mean(list(roc_auc.values())):.4f}")
    print("="*60)

    return summary

# Run comprehensive evaluation
if len(val_accs) > 0:  # Only run if we have training history
    print("Starting comprehensive evaluation...")
    summary = create_comprehensive_report(model, test_loader, device, class_names)
else:
    print("No training history available for evaluation.")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, classification_report

def calculate_comprehensive_metrics(true_labels, predictions, class_names):
    """Calculate comprehensive precision, recall, F1 metrics with detailed analysis"""

    # Calculate per-class metrics
    precision, recall, f1, support = precision_recall_fscore_support(
        true_labels, predictions, average=None, zero_division=0
    )

    # Calculate macro averages
    precision_macro = np.mean(precision)
    recall_macro = np.mean(recall)
    f1_macro = np.mean(f1)

    # Calculate weighted averages
    precision_weighted = np.average(precision, weights=support)
    recall_weighted = np.average(recall, weights=support)
    f1_weighted = np.average(f1, weights=support)

    # Calculate micro average (equivalent to accuracy for multi-class)
    accuracy = np.mean(true_labels == predictions)

    # Create detailed per-class dataframe
    per_class_metrics = []
    for i, class_name in enumerate(class_names):
        per_class_metrics.append({
            'Class': class_name,
            'Precision': f"{precision[i]:.4f}",
            'Recall': f"{recall[i]:.4f}",
            'F1-Score': f"{f1[i]:.4f}",
            'Support': support[i]
        })

    # Add overall metrics
    overall_metrics = [
        {
            'Class': 'Macro Average',
            'Precision': f"{precision_macro:.4f}",
            'Recall': f"{recall_macro:.4f}",
            'F1-Score': f"{f1_macro:.4f}",
            'Support': np.sum(support)
        },
        {
            'Class': 'Weighted Average',
            'Precision': f"{precision_weighted:.4f}",
            'Recall': f"{recall_weighted:.4f}",
            'F1-Score': f"{f1_weighted:.4f}",
            'Support': np.sum(support)
        }
    ]

    metrics_df = pd.DataFrame(per_class_metrics + overall_metrics)

    return metrics_df, precision, recall, f1, support

def print_detailed_analysis(precision, recall, f1, support, class_names):
    """Print detailed analysis of the metrics"""

    print("\n" + "="*80)
    print("📊 DETAILED PRECISION, RECALL, F1-SCORE ANALYSIS")
    print("="*80)

    # Best performing classes
    best_f1_idx = np.argmax(f1)
    best_precision_idx = np.argmax(precision)
    best_recall_idx = np.argmax(recall)

    print(f"\n🏆 BEST PERFORMING CLASSES:")
    print(f"  • Highest F1-Score: {class_names[best_f1_idx]} = {f1[best_f1_idx]:.4f}")
    print(f"  • Highest Precision: {class_names[best_precision_idx]} = {precision[best_precision_idx]:.4f}")
    print(f"  • Highest Recall: {class_names[best_recall_idx]} = {recall[best_recall_idx]:.4f}")

    # Worst performing classes
    worst_f1_idx = np.argmin(f1)
    worst_precision_idx = np.argmin(precision)
    worst_recall_idx = np.argmin(recall)

    print(f"\n⚠️  CHALLENGING CLASSES:")
    print(f"  • Lowest F1-Score: {class_names[worst_f1_idx]} = {f1[worst_f1_idx]:.4f}")
    print(f"  • Lowest Precision: {class_names[worst_precision_idx]} = {precision[worst_precision_idx]:.4f}")
    print(f"  • Lowest Recall: {class_names[worst_recall_idx]} = {recall[worst_recall_idx]:.4f}")

    # Performance gaps
    f1_range = f1.max() - f1.min()
    precision_range = precision.max() - precision.min()
    recall_range = recall.max() - recall.min()

    print(f"\n📈 PERFORMANCE RANGES:")
    print(f"  • F1-Score Range: {f1_range:.4f} ({f1.min():.4f} - {f1.max():.4f})")
    print(f"  • Precision Range: {precision_range:.4f} ({precision.min():.4f} - {precision.max():.4f})")
    print(f"  • Recall Range: {recall_range:.4f} ({recall.min():.4f} - {recall.max():.4f})")

    # Class balance analysis
    support_ratio = support.max() / support.min() if support.min() > 0 else float('inf')
    print(f"\n⚖️  CLASS BALANCE:")
    print(f"  • Most samples: {class_names[np.argmax(support)]} = {support.max()}")
    print(f"  • Fewest samples: {class_names[np.argmin(support)]} = {support.min()}")
    print(f"  • Support ratio: {support_ratio:.2f}x")

    # Quality indicators
    high_precision_classes = np.sum(precision > 0.8)
    high_recall_classes = np.sum(recall > 0.8)
    high_f1_classes = np.sum(f1 > 0.8)

    print(f"\n🎯 QUALITY INDICATORS:")
    print(f"  • Classes with Precision > 0.80: {high_precision_classes}/{len(class_names)}")
    print(f"  • Classes with Recall > 0.80: {high_recall_classes}/{len(class_names)}")
    print(f"  • Classes with F1-Score > 0.80: {high_f1_classes}/{len(class_names)}")

def get_all_predictions(model, dataloader, device):
    """Get all predictions and true labels"""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            probs = F.softmax(outputs["emotion"], dim=1)
            _, preds = torch.max(outputs["emotion"], 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

# Main evaluation function
def evaluate_precision_recall_f1(model, test_loader, device, class_names):
    """Comprehensive precision, recall, F1 evaluation"""

    print("🚀 Calculating Precision, Recall, F1-Score...")

    # Get predictions
    true_labels, predictions, probabilities = get_all_predictions(model, test_loader, device)

    # Calculate metrics
    metrics_df, precision, recall, f1, support = calculate_comprehensive_metrics(
        true_labels, predictions, class_names
    )

    # Print detailed table
    print("\n" + "="*80)
    print("📋 PRECISION, RECALL, F1-SCORE RESULTS")
    print("="*80)
    print(metrics_df.to_string(index=False))

    # Print detailed analysis
    print_detailed_analysis(precision, recall, f1, support, class_names)

    # Additional metrics from classification report
    print("\n" + "="*80)
    print("📄 COMPLETE CLASSIFICATION REPORT")
    print("="*80)
    print(classification_report(true_labels, predictions, target_names=class_names, digits=4))

    # Save to file
    metrics_df.to_csv(os.path.join(save_dir, 'precision_recall_f1_metrics.csv'), index=False)

    # Create summary
    summary = {
        'macro_precision': np.mean(precision),
        'macro_recall': np.mean(recall),
        'macro_f1': np.mean(f1),
        'weighted_precision': np.average(precision, weights=support),
        'weighted_recall': np.average(recall, weights=support),
        'weighted_f1': np.average(f1, weights=support),
        'accuracy': np.mean(true_labels == predictions),
        'best_class': class_names[np.argmax(f1)],
        'worst_class': class_names[np.argmin(f1)],
        'f1_range': f1.max() - f1.min()
    }

    print("\n" + "="*80)
    print("🎯 KEY SUMMARY METRICS")
    print("="*80)
    print(f"Overall Accuracy: {summary['accuracy']:.4f}")
    print(f"Macro Precision:  {summary['macro_precision']:.4f}")
    print(f"Macro Recall:     {summary['macro_recall']:.4f}")
    print(f"Macro F1-Score:   {summary['macro_f1']:.4f}")
    print(f"Weighted F1:      {summary['weighted_f1']:.4f}")
    print(f"F1-Score Range:   {summary['f1_range']:.4f}")
    print(f"Best Class:       {summary['best_class']} (F1 = {f1.max():.4f})")
    print(f"Most Challenging: {summary['worst_class']} (F1 = {f1.min():.4f})")

    # Save summary
    with open(os.path.join(save_dir, 'metrics_summary.json'), 'w') as f:
        json.dump(summary, f, indent=2)

    return summary, metrics_df

# Run the evaluation
print("Starting precision, recall, F1-score evaluation...")
summary, metrics_df = evaluate_precision_recall_f1(model, test_loader, device, class_names)

print(f"\n✅ Evaluation complete! Results saved to {save_dir}")